# Eye-Tracking: Gaze Offset Correction

Remote eye-trackers often produce a systematic **spatial offset**: the recorded gaze position
is consistently shifted from where the participant was actually looking. This is caused by
individual differences in head position, facial anatomy, and device calibration quality.

This notebook implements and **compares two correction strategies**:

| Method | Assumption | Best for |
|--------|------------|----------|
| **Static** | Offset is constant across the entire session | Short sessions, stable head position |
| **Dynamic** | Offset drifts slowly across trials (rolling average) | Long sessions, children, remote trackers |

Both methods assume that stimuli are **spatially balanced**: averaged over all trials,
the expected gaze position should equal the theoretical centre of the stimulus layout.

---

## Workflow
1. Define AOIs and compute theoretical gaze centre.
2. Load data.
3. Compute **static** correction vectors (one per participant).
4. Compute **dynamic** correction vectors (one per participant x trial, smoothed).
5. Apply static correction (recommended default).
6. Validate with simple heatmap comparison.
7. Validate with topographic comparison (static vs dynamic).
8. Save corrected data.

---

## Data format expected

```python
aggregated_data = {
    'participant_id': {
        'info':   {'group': 'A', ...},
        'trials': {
            1: {'points': [{'x': float, 'y': float, 't': float}, ...]},
            2: {'points': [...]},
        }
    }
}
```

All `x`, `y` values must be **normalised to [0, 1]**. `t` is in milliseconds.

---

## References
- Holmqvist, K., et al. (2011). *Eye Tracking: A Comprehensive Guide*. OUP.
- Blignaut, P. (2009). Fixation identification: The optimum threshold for a dispersion algorithm. *APP*, 71, 881-895.
- Nyström, M., & Holmqvist, K. (2010). An adaptive algorithm for fixation detection. *BRM*, 42, 188-204.

## 0. Setup

In [ ]:
import copy
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as colors
from matplotlib.gridspec import GridSpec
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

print('Libraries loaded.')

## 1. Configuration

All user-facing parameters are collected here. **Edit this cell** before running the notebook.

In [ ]:
# ---------------------------------------------------------------------------
# AOI LAYOUT
# Keys: AOI labels. Values: (x_min, y_min, x_max, y_max) in normalised [0,1].
# Adapt to your stimulus layout.
# ---------------------------------------------------------------------------
AOIS = {
    # Problem matrix (4 cells)
    'A':  (0.25, 0.025, 0.49, 0.255),
    'B':  (0.49, 0.025, 0.75, 0.255),
    'C':  (0.25, 0.255, 0.49, 0.485),
    'D':  (0.49, 0.255, 0.75, 0.485),
    # Response options (6 cells)
    'R1': (0.20, 0.535, 0.39, 0.770),
    'R2': (0.39, 0.535, 0.58, 0.770),
    'R3': (0.58, 0.535, 0.77, 0.770),
    'R4': (0.20, 0.770, 0.39, 0.985),
    'R5': (0.39, 0.770, 0.58, 0.985),
    'R6': (0.58, 0.770, 0.77, 0.985),
}

# ---------------------------------------------------------------------------
# STATIC CORRECTION
# ---------------------------------------------------------------------------
MIN_POINTS_FOR_CORRECTION = 50   # Skip participants with fewer total samples.

# ---------------------------------------------------------------------------
# DYNAMIC CORRECTION
# ---------------------------------------------------------------------------
DYNAMIC_WINDOW_SIZE = 11         # Rolling average window (number of trials).
                                  # Smaller -> more responsive to drift.
                                  # Larger  -> smoother, less sensitive.
                                  # Rule of thumb: ~25-30% of total trials.

# ---------------------------------------------------------------------------
# TOPOGRAPHIC VALIDATION
# ---------------------------------------------------------------------------
TARGET_AOIS_FOR_VIZ = ['R1', 'R3']  # AOIs used for topographic comparison.
                                      # Choose spatially separated AOIs.
AOI_MARGIN_PERCENT  = 0.10           # Expand AOI boundaries by this fraction
                                      # before collecting samples (avoids edge
                                      # effects near AOI boundaries).

# ---------------------------------------------------------------------------
# Derived constant — do not edit
# ---------------------------------------------------------------------------
aoi_centres = np.array([
    [xmin + (xmax - xmin) / 2, ymin + (ymax - ymin) / 2]
    for xmin, ymin, xmax, ymax in AOIS.values()
])
THEORETICAL_CENTRE = aoi_centres.mean(axis=0)

print(f'Defined {len(AOIS)} AOIs.')
print(f'Theoretical gaze centre: x={THEORETICAL_CENTRE[0]:.3f},  y={THEORETICAL_CENTRE[1]:.3f}')
print(f'Dynamic correction window: {DYNAMIC_WINDOW_SIZE} trials')

### 1.1 Visualise AOI layout

In [ ]:
def plot_aoi_layout(aois, theoretical_centre):
    """Draw all AOIs as labelled rectangles on a normalised [0,1] canvas."""
    fig, ax = plt.subplots(figsize=(6, 7))
    ax.set_xlim(0, 1)
    ax.set_ylim(1, 0)   # flip y: (0,0) = top-left, matching screen coordinates
    ax.set_xlabel('x (normalised)')
    ax.set_ylabel('y (normalised)')
    ax.set_title('AOI Layout')

    for name, (x0, y0, x1, y1) in aois.items():
        ax.add_patch(patches.Rectangle(
            (x0, y0), x1 - x0, y1 - y0,
            linewidth=1.5, edgecolor='steelblue',
            facecolor='lightsteelblue', alpha=0.4
        ))
        ax.text((x0+x1)/2, (y0+y1)/2, name,
                ha='center', va='center', fontsize=10,
                fontweight='bold', color='navy')

    ax.scatter(*theoretical_centre, color='red', zorder=5, s=80,
               label=f'Theoretical centre ({theoretical_centre[0]:.2f}, {theoretical_centre[1]:.2f})')
    ax.legend()
    plt.tight_layout()
    plt.show()


plot_aoi_layout(AOIS, THEORETICAL_CENTRE)

## 2. Load data

Replace the cell below with your own loading code.  
A synthetic example is provided so the notebook runs end-to-end without real data.

In [ ]:
# ---------------------------------------------------------------------------
# REPLACE THIS CELL with your data loading logic, e.g.:
#
#   from your_pipeline import aggregate_raw_data
#   aggregated_data = aggregate_raw_data(list_of_json_files)
#
# The only requirement: produce a dict matching the structure in the header.
# ---------------------------------------------------------------------------

# Synthetic data for demonstration
# Each participant has a static offset + slow linear drift across trials
rng = np.random.default_rng(42)

def _make_participant(pid, offset, n_trials=36, drift_per_trial=0.001):
    """
    Simulate a participant with static offset and slow linear drift.
    drift_per_trial mimics real within-session tracker drift.
    """
    trials = {}
    ox, oy = offset
    for t in range(1, n_trials + 1):
        cx = float(np.clip(THEORETICAL_CENTRE[0] + ox + drift_per_trial * t, 0.05, 0.95))
        cy = float(np.clip(THEORETICAL_CENTRE[1] + oy + drift_per_trial * t, 0.05, 0.95))
        points = [
            {'x': float(np.clip(cx + rng.normal(0, 0.025), 0, 1)),
             'y': float(np.clip(cy + rng.normal(0, 0.025), 0, 1)),
             't': float(k * 16.67)}
            for k in range(60)
        ]
        trials[t] = {'points': points}
    return {'info': {'group': 'A' if int(pid[1:]) % 2 == 0 else 'B'}, 'trials': trials}

aggregated_data = {
    'P01': _make_participant('P01', (-0.06,  0.04),  drift_per_trial= 0.001),
    'P02': _make_participant('P02', ( 0.03, -0.05),  drift_per_trial=-0.001),
    'P03': _make_participant('P03', ( 0.00,  0.00),  drift_per_trial= 0.000),
    'P04': _make_participant('P04', ( 0.07,  0.06),  drift_per_trial= 0.002),
    'P05': _make_participant('P05', (-0.04, -0.03),  drift_per_trial=-0.001),
    'P06': _make_participant('P06', ( 0.02,  0.05),  drift_per_trial= 0.001),
}

n_total = sum(len(t['points']) for p in aggregated_data.values() for t in p['trials'].values())
print(f'Loaded {len(aggregated_data)} participants, '
      f'{sum(len(p["trials"]) for p in aggregated_data.values())} trials, '
      f'{n_total:,} gaze samples.')

## 3. Static correction

One correction vector per participant, computed from the session-wide mean gaze position:

$$\vec{v}_i = \bar{c}_{\text{AOI}} - \bar{g}_i$$

where $\bar{g}_i$ is the participant's mean gaze across all trials and $\bar{c}_{\text{AOI}}$ is the theoretical centre.

In [ ]:
def compute_static_vectors(data, theoretical_centre, min_points=MIN_POINTS_FOR_CORRECTION):
    """
    Compute a per-participant static offset correction vector.

    The vector shifts the participant's session-wide mean gaze onto the
    theoretical centre. Participants with fewer than min_points samples
    are skipped.

    Parameters
    ----------
    data               : aggregated gaze data dict.
    theoretical_centre : (2,) array, expected mean gaze position.
    min_points         : minimum total samples required.

    Returns
    -------
    dict: {participant_id -> np.ndarray([dx, dy])}
    """
    vectors = {}
    skipped = []

    for pid, pdata in data.items():
        all_xy = [
            (p['x'], p['y'])
            for trial in pdata['trials'].values()
            for p in trial['points']
        ]
        if len(all_xy) < min_points:
            skipped.append(pid)
            continue
        vectors[pid] = theoretical_centre - np.mean(all_xy, axis=0)

    if skipped:
        print(f'  Skipped (< {min_points} samples): {skipped}')
    print(f'Static vectors computed for {len(vectors)} participants.')
    return vectors


static_vectors = compute_static_vectors(aggregated_data, THEORETICAL_CENTRE)

df_static = pd.DataFrame(
    [(pid, v[0], v[1], float(np.linalg.norm(v))) for pid, v in static_vectors.items()],
    columns=['participant', 'dx', 'dy', 'magnitude']
).set_index('participant')

print('\nMagnitude guide: <0.02 negligible | 0.02-0.05 moderate | >0.05 large')
display(df_static.style.format('{:.4f}').background_gradient(cmap='YlOrRd', subset=['magnitude']))

## 4. Dynamic correction

Within a long session, tracker offset can **drift** as participants shift posture or the
tracker loses optimal calibration. The dynamic method captures this by computing a
**rolling average** of the per-trial centre of mass over a window of `DYNAMIC_WINDOW_SIZE` trials:

$$\vec{v}_{i,t} = \bar{c}_{\text{AOI}} - \overline{\text{CoM}}_{i,\, t \pm w/2}$$

This smooths out genuine fixation variance while tracking slow systematic drift.

In [ ]:
def compute_com_per_trial(data):
    """
    Compute the centre of mass (CoM) of gaze for each participant x trial.

    This is the per-trial equivalent of the session-wide mean used for
    static correction, and is the input to the rolling-average step.

    Parameters
    ----------
    data : aggregated gaze data dict.

    Returns
    -------
    pd.DataFrame with columns ['participant_id', 'trial_num', 'com_x', 'com_y'].
    """
    rows = []
    for pid, pdata in data.items():
        for trial_num, trial_data in pdata['trials'].items():
            pts = [(p['x'], p['y']) for p in trial_data['points']]
            if len(pts) > 10:
                com = np.mean(pts, axis=0)
                rows.append({'participant_id': pid, 'trial_num': trial_num,
                              'com_x': com[0], 'com_y': com[1]})
    return pd.DataFrame(rows)


def compute_dynamic_vectors(df_com, theoretical_centre, window_size=DYNAMIC_WINDOW_SIZE):
    """
    Compute per-trial dynamic correction vectors using a rolling average.

    For each participant, the per-trial centre of mass is smoothed with a
    centred rolling window of `window_size` trials. The correction vector
    for trial t is the difference between the theoretical centre and the
    smoothed CoM at t.

    A larger window tracks only slow drift; a smaller window responds faster
    but risks overcorrecting genuine behavioural variation.

    Parameters
    ----------
    df_com             : output of compute_com_per_trial().
    theoretical_centre : (2,) array, expected mean gaze position.
    window_size        : rolling window in number of trials.
                         Rule of thumb: ~25-30% of total trials.

    Returns
    -------
    defaultdict: {participant_id: {trial_num: np.ndarray([dx, dy])}}
    """
    if df_com.empty:
        print('WARNING: empty CoM DataFrame.')
        return defaultdict(dict)

    df = df_com.sort_values(['participant_id', 'trial_num']).copy()

    for axis, idx in [('x', 0), ('y', 1)]:
        rolling = df.groupby('participant_id')[f'com_{axis}'].transform(
            lambda s: s.rolling(window=window_size, center=True, min_periods=1).mean()
        )
        df[f'dynamic_vec_{axis}'] = theoretical_centre[idx] - rolling

    result = defaultdict(dict)
    for _, row in df.iterrows():
        result[row['participant_id']][row['trial_num']] = np.array(
            [row['dynamic_vec_x'], row['dynamic_vec_y']]
        )

    print(f'Dynamic vectors computed (window = {window_size} trials).')
    return result


df_com = compute_com_per_trial(aggregated_data)
dynamic_vectors = compute_dynamic_vectors(df_com, THEORETICAL_CENTRE)
print(f'Total trials with dynamic vectors: {sum(len(v) for v in dynamic_vectors.values())}')

### 4.1 Visualise drift across trials

The thin line is the raw per-trial CoM; the thick line is the rolling average.
Large divergences between them indicate genuine behavioural variation — which the
rolling average correctly ignores.

In [ ]:
def plot_drift(df_com, window_size=DYNAMIC_WINDOW_SIZE):
    """Plot per-trial CoM with rolling average overlay for each participant."""
    participants = df_com['participant_id'].unique()
    n = len(participants)
    fig, axes = plt.subplots(n, 2, figsize=(14, 2.8 * n), sharex=True)
    if n == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(
        f'Per-trial gaze CoM: raw vs rolling average (window={window_size})',
        fontsize=13, y=1.01
    )

    for i, pid in enumerate(participants):
        df_p = df_com[df_com['participant_id'] == pid].sort_values('trial_num')
        for j, (col, ref) in enumerate([('com_x', THEORETICAL_CENTRE[0]),
                                         ('com_y', THEORETICAL_CENTRE[1])]):
            ax = axes[i, j]
            rolling = df_p[col].rolling(window=window_size, center=True, min_periods=1).mean()

            ax.plot(df_p['trial_num'], df_p[col],
                    color='steelblue', alpha=0.35, linewidth=1, label='Raw CoM')
            ax.plot(df_p['trial_num'], rolling,
                    color='steelblue', linewidth=2.2,
                    label=f'Rolling avg (w={window_size})')
            ax.axhline(ref, color='red', linestyle='--', linewidth=1.2,
                       label='Theoretical centre')

            ax.set_ylabel(f'Gaze centre {col[-1].upper()}', fontsize=9)
            ax.set_title(pid, fontsize=9, loc='left')
            if i == 0 and j == 0:
                ax.legend(fontsize=8)

    axes[-1, 0].set_xlabel('Trial')
    axes[-1, 1].set_xlabel('Trial')
    plt.tight_layout()
    plt.show()


plot_drift(df_com)

## 5. Apply static correction

The static correction is the **recommended default**: simpler, more interpretable, and
less likely to overcorrect genuine behavioural differences between trials.  
Use the dynamic correction only if the drift plot above shows a clear systematic trend.

In [ ]:
def apply_static_correction(data, vectors):
    """
    Apply per-participant static offset correction to all gaze samples.

    Creates and returns a deep copy of `data` — the original is not modified.
    Corrected coordinates are clipped to [0, 1].

    Parameters
    ----------
    data    : aggregated gaze data dict.
    vectors : {participant_id -> np.ndarray([dx, dy])}

    Returns
    -------
    Deep copy of data with corrected gaze coordinates.
    """
    corrected = copy.deepcopy(data)
    for pid, pdata in corrected.items():
        vec = vectors.get(pid)
        if vec is None:
            continue
        for trial in pdata['trials'].values():
            for p in trial['points']:
                p['x'] = float(np.clip(p['x'] + vec[0], 0.0, 1.0))
                p['y'] = float(np.clip(p['y'] + vec[1], 0.0, 1.0))
    return corrected


corrected_data = apply_static_correction(aggregated_data, static_vectors)
print('Static correction applied. Original `aggregated_data` is unchanged.')

## 6. Validation — heatmap comparison

2D density histogram of all gaze samples pooled across participants, before and after
correction. The centre of mass (cyan x) should move closer to the theoretical centre
(red +) after correction.

In [ ]:
def _collect_xy(data):
    xs, ys = [], []
    for pdata in data.values():
        for trial in pdata['trials'].values():
            for p in trial['points']:
                xs.append(p['x']); ys.append(p['y'])
    return xs, ys


def plot_heatmap_comparison(raw_data, corrected_data, aois, theoretical_centre):
    """
    Two-panel heatmap: raw gaze density vs corrected gaze density.

    Both panels share the same log-normalised colour scale.
    AOI outlines, observed centre of mass, and theoretical centre are overlaid.
    """
    raw_x,  raw_y  = _collect_xy(raw_data)
    cor_x,  cor_y  = _collect_xy(corrected_data)

    vmax = max(
        np.histogram2d(raw_x, raw_y, bins=50, range=[[0,1],[0,1]])[0].max(),
        np.histogram2d(cor_x, cor_y, bins=50, range=[[0,1],[0,1]])[0].max(),
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=True, sharey=True)
    fig.suptitle('Offset correction: gaze density before and after',
                 fontsize=15, weight='bold')

    for ax, xs, ys, title in zip(
        axes,
        [raw_x, cor_x], [raw_y, cor_y],
        ['Before correction (raw)', 'After static correction']
    ):
        h = ax.hist2d(xs, ys, bins=50, range=[[0,1],[0,1]],
                      cmap='inferno', norm=colors.LogNorm(vmin=1, vmax=vmax))

        for _, (x0, y0, x1, y1) in aois.items():
            ax.add_patch(patches.Rectangle(
                (x0, y0), x1-x0, y1-y0,
                linewidth=1.2, edgecolor='lime', facecolor='none', alpha=0.8
            ))

        com_x, com_y = np.mean(xs), np.mean(ys)
        ax.plot(com_x, com_y, 'x', color='cyan', markersize=14, markeredgewidth=2.5,
                label=f'CoM ({com_x:.3f}, {com_y:.3f})')
        ax.scatter(*theoretical_centre, color='red', marker='+', s=200,
                   linewidths=2, zorder=5,
                   label=f'Theoretical ({theoretical_centre[0]:.3f}, {theoretical_centre[1]:.3f})')

        ax.set_title(title, fontsize=13)
        ax.set_xlabel('x (normalised)')
        ax.set_aspect('equal')
        ax.legend(loc='upper right', fontsize=9)

    axes[0].set_ylabel('y (normalised)')
    axes[0].invert_yaxis()
    fig.colorbar(h[3], ax=axes, orientation='vertical',
                 fraction=0.03, pad=0.04, label='Sample density (log scale)')
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


plot_heatmap_comparison(aggregated_data, corrected_data, AOIS, THEORETICAL_CENTRE)

## 7. Validation — topographic comparison (static vs dynamic)

To compare correction quality more precisely, gaze samples from `TARGET_AOIS_FOR_VIZ`
are **re-centred** into a common normalised space where `(0.5, 0.5)` is the ideal AOI
centre. This allows direct comparison across screen positions and across methods.

The **centring error** (Euclidean distance from CoM to `(0.5, 0.5)`) quantifies
correction quality: lower is better.

In [ ]:
def collect_recentered_points(
    data, static_vectors, dynamic_vectors, aois,
    target_aois=TARGET_AOIS_FOR_VIZ,
    margin=AOI_MARGIN_PERCENT,
):
    """
    Collect gaze samples from target AOIs and re-centre them into a common
    normalised space for topographic comparison.

    For each sample inside a target AOI (expanded by `margin`), three versions
    are stored: raw, statically corrected, and dynamically corrected. Each is
    re-centred so that the AOI centre maps to (0.5, 0.5), allowing AOIs from
    different screen positions to be directly overlaid.

    Parameters
    ----------
    data            : raw (uncorrected) aggregated gaze data.
    static_vectors  : output of compute_static_vectors().
    dynamic_vectors : output of compute_dynamic_vectors().
    aois            : AOI layout dict.
    target_aois     : AOI labels to use (choose spatially separated ones).
    margin          : fractional expansion of AOI boundaries (avoids edge effects).

    Returns
    -------
    dict: {'raw': [...], 'static': [...], 'dynamic': [...]}
    Each list contains (x, y) tuples in re-centred coordinates.
    """
    # Expand AOI boundaries
    padded = {}
    for name in target_aois:
        x0, y0, x1, y1 = aois[name]
        pw, ph = (x1 - x0) * margin, (y1 - y0) * margin
        padded[name] = (x0 - pw, y0 - ph, x1 + pw, y1 + ph)

    pts = {'raw': [], 'static': [], 'dynamic': []}

    for pid, pdata in data.items():
        sv = static_vectors.get(pid, np.zeros(2))

        for trial_num, trial_data in pdata['trials'].items():
            dv = dynamic_vectors.get(pid, {}).get(trial_num, sv)

            for p in trial_data['points']:
                x, y = p.get('x'), p.get('y')
                if x is None or y is None:
                    continue

                for aoi_name, (px0, py0, px1, py1) in padded.items():
                    if not (px0 <= x <= px1 and py0 <= y <= py1):
                        continue
                    pw, ph = px1 - px0, py1 - py0
                    if pw == 0 or ph == 0:
                        continue

                    # Raw: re-centre relative to padded AOI
                    pts['raw'].append(((x - px0)/pw, (y - py0)/ph))

                    # Static correction
                    xs = float(np.clip(x + sv[0], 0, 1))
                    ys = float(np.clip(y + sv[1], 0, 1))
                    pts['static'].append(((xs - px0)/pw, (ys - py0)/ph))

                    # Dynamic correction
                    xd = float(np.clip(x + dv[0], 0, 1))
                    yd = float(np.clip(y + dv[1], 0, 1))
                    pts['dynamic'].append(((xd - px0)/pw, (yd - py0)/ph))

                    break  # each point belongs to at most one AOI

    for k, v in pts.items():
        print(f'  {k:8s}: {len(v):,} re-centred samples')
    return pts


recentered = collect_recentered_points(
    aggregated_data, static_vectors, dynamic_vectors, AOIS
)

In [ ]:
def plot_topographic_comparison(recentered):
    """
    Three-panel topographic comparison: no correction / static / dynamic.

    Each panel shows:
    - 2D density heatmap of re-centred gaze samples.
    - KDE contours at 50th / 68th / 95th percentile density regions.
    - Observed CoM (white x) and centring error.
    - White box: AOI boundary (without margin).
    - Cyan crosshairs at ideal centre (0.5, 0.5).

    Centring error = distance from CoM to (0.5, 0.5). Lower is better.
    """
    conditions = [
        ('raw',     '1. No correction'),
        ('static',  '2. Static correction'),
        ('dynamic', '3. Dynamic correction'),
    ]

    all_counts = [
        np.histogram2d(*zip(*recentered[k]), bins=50, range=[[0,1],[0,1]])[0]
        for k, _ in conditions if recentered[k]
    ]
    vmax = max(c.max() for c in all_counts) if all_counts else 1

    fig = plt.figure(figsize=(22, 7))
    gs  = GridSpec(1, 22, figure=fig)
    axes = [
        fig.add_subplot(gs[0, 0:6]),
        fig.add_subplot(gs[0, 7:13]),
        fig.add_subplot(gs[0, 14:20]),
    ]
    cax = fig.add_subplot(gs[0, 21])

    fig.suptitle(
        f'Topographic validation — AOIs: {TARGET_AOIS_FOR_VIZ}  '
        f'(margin={int(AOI_MARGIN_PERCENT*100)}%)\n'
        f'Ideal gaze centre = (0.5, 0.5).',
        fontsize=13, weight='bold'
    )

    h_last = None
    for ax, (key, title) in zip(axes, conditions):
        pts = recentered[key]
        if not pts:
            ax.set_title(f'{title}\n(no data)', fontsize=12)
            continue

        xs, ys = zip(*pts)
        h_last = ax.hist2d(
            xs, ys, bins=50, range=[[0,1],[0,1]],
            cmap='inferno', norm=colors.LogNorm(vmin=1, vmax=vmax)
        )

        try:
            sns.kdeplot(
                x=xs, y=ys, ax=ax,
                levels=[0.05, 0.32, 0.50],
                color='white',
                line_kws={'linewidths': [1.0, 1.5, 2.0],
                          'linestyles': ['--', '-', ':']}
            )
        except Exception:
            pass

        com_x, com_y = np.mean(xs), np.mean(ys)
        error = np.linalg.norm([com_x - 0.5, com_y - 0.5])

        ax.plot(com_x, com_y, 'x', color='white', markersize=13,
                markeredgewidth=2.5, zorder=6)
        ax.text(0.04, 0.96,
                f'Centring error: {error:.4f}\nCoM: ({com_x:.3f}, {com_y:.3f})',
                transform=ax.transAxes, fontsize=10, color='white',
                verticalalignment='top',
                bbox=dict(boxstyle='round,pad=0.3', fc='black', alpha=0.55))

        ax.axhline(0.5, color='cyan', linestyle='--', linewidth=1.3, alpha=0.8)
        ax.axvline(0.5, color='cyan', linestyle='--', linewidth=1.3, alpha=0.8)

        m = AOI_MARGIN_PERCENT
        box_start = m / (1 + 2 * m)
        box_size  = 1 / (1 + 2 * m)
        ax.add_patch(patches.Rectangle(
            (box_start, box_start), box_size, box_size,
            fill=False, edgecolor='white', linewidth=2
        ))

        ax.set_title(title, fontsize=13)
        ax.set_xlabel('x (re-centred)')
        ax.set_aspect('equal')

    axes[0].set_ylabel('y (re-centred)')
    if h_last:
        fig.colorbar(h_last[3], cax=cax, label='Sample density (log scale)')
    plt.tight_layout(rect=[0, 0, 1, 0.92])
    plt.show()


plot_topographic_comparison(recentered)

### 7.1 Summary table — centring error by method

In [ ]:
rows = []
for key, label in [('raw','No correction'), ('static','Static'), ('dynamic','Dynamic')]:
    pts = recentered[key]
    if pts:
        xs, ys = zip(*pts)
        com = np.array([np.mean(xs), np.mean(ys)])
        rows.append({
            'Method': label,
            'CoM x': com[0],
            'CoM y': com[1],
            'Centring error': float(np.linalg.norm(com - 0.5)),
            'n samples': len(pts),
        })

df_summary = pd.DataFrame(rows).set_index('Method')
display(
    df_summary.style
    .format({'CoM x': '{:.4f}', 'CoM y': '{:.4f}', 'Centring error': '{:.4f}'})
    .background_gradient(cmap='RdYlGn_r', subset=['Centring error'])
)

## 8. Save corrected data

In [ ]:
import json, os

# Set to a file path to save, or None to keep in memory only
OUTPUT_PATH = None   # e.g. '../data/corrected_gaze.json'

if OUTPUT_PATH:
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    def _serialise(obj):
        if isinstance(obj, np.ndarray): return obj.tolist()
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, (np.integer,)):  return int(obj)
        raise TypeError(f'Not serialisable: {type(obj)}')

    with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
        json.dump(corrected_data, f, default=_serialise, indent=2)
    print(f'Saved to {OUTPUT_PATH}')
else:
    print('OUTPUT_PATH is None — corrected_data kept in memory.')
    print('Pass `corrected_data` directly to 02_fixation_analysis.ipynb.')

## Summary

| Variable | Contents |
|----------|----------|
| `static_vectors`  | Per-participant correction vectors (session-wide mean) |
| `dynamic_vectors` | Per-participant x per-trial vectors (rolling average, window=`DYNAMIC_WINDOW_SIZE`) |
| `corrected_data`  | Deep copy of input data with static correction applied |
| `df_com`          | Per-trial centre of mass (useful for drift diagnostics) |

**Which method to use?** If the centring errors in Section 7.1 are similar, prefer static
(simpler, more reproducible). If dynamic is substantially better and the drift plot shows a
clear trend, switch to dynamic by re-running Section 5 with `apply_static_correction` replaced
by an equivalent dynamic version.

➡️ Continue with **`02_fixation_analysis.ipynb`** to run the I-DT fixation detection algorithm on `corrected_data`.